In [2]:
import sys

sys.path.append('..')

In [ ]:
import numpy as np
from transformers import pipeline
import gradio as gr
import os
import time
import json
import pika
from serde.json import from_json
from utils import add_to_queue, read_from_queue, speech_to_text_call, get_rmq_channel, ChatQuestion, \
    VideoResponse, SubtitleMode
from character_setup import CHARACTERS



character = CHARACTERS["Jesus_de"].name
history = []
prompt = """
    You are a german teacher. You are roleplaying with your student as Jesus to help him learn. 
    The level of your student is B1 so you have to make simple sentence but 
    you should create some content to keep the conversation going. 
    You can also precise some grammatical or lexical points.
    ```
    {history}
    {name}: {question}
    Jesus: 
    ```
    Only generate what Jesus would say.
    """
query_text = "Hallo Jesus, wie geht's ?"

server_queue = "chat_log"


channel = get_rmq_channel(server_queue)
# The answer channel must be prepared
next(channel.consume(queue="amq.rabbitmq.reply-to", auto_ack=True, inactivity_timeout=0.1))

def dummy_query():
    # 2 - Get the query and send it
    query_text = ""
    q = ChatQuestion("GentilUtilisateur", query_text, None, prompt, history,
                     character_name=character, subtitle_mode=SubtitleMode.NONE)
    text = q.serialize()
    print("Text content : ", text)
    channel.basic_publish(
        exchange="", routing_key=server_queue, body=text.encode(),
        properties=pika.BasicProperties(reply_to="amq.rabbitmq.reply-to"))
    print("sent:", text)

    # 3 - Wait for the response from server
    (method, properties, body) = next(channel.consume(queue="amq.rabbitmq.reply-to", auto_ack=True,
                                                      inactivity_timeout=2*60))
    return from_json(VideoResponse, body.decode())
    

In [ ]:
dummy_query()